In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path
import pandas as pd

In [3]:
# set up relative imports
project_folder = Path.cwd().parent.parent
sys.path.append(str(project_folder))

In [4]:
from app.fetch_data import fetch_daily_buoy_data, fetch_daily_weather, fetch_water_info


In [22]:
weather_df = fetch_daily_weather()
water_df = fetch_water_info()
wind_df = fetch_daily_buoy_data()

date =  2026-07-30 00:00:00
surface temp =  21.0
bottom temp =  18.2


In [23]:
weather_df

,date,min,max,PRCP
0,2026-07-27,70,71.1,0
1,2026-07-28,69.1,77,0
2,2026-07-29,69.1,77,0
3,2026-07-30,68,77,0


In [24]:
water_df

,date,surf_temp_c,bottom_temp
0,2026-07-30,21.0,18.2


In [25]:
wind_df

,date,WSPD,WDIR
0,2026-06-16,3.125523,316.192469
1,2026-06-17,2.839241,283.628692
2,2026-06-18,2.455357,248.616071
3,2026-06-19,2.877500,279.000000
4,2026-06-20,3.446383,297.234043
5,2026-06-21,3.031760,286.051502
6,2026-06-22,2.660086,283.218884
7,2026-06-23,2.090741,259.907407
8,2026-06-24,2.092174,284.086957
9,2026-06-25,2.664192,272.227074


In [26]:
merge_df = weather_df.merge(wind_df,
                            how='left',
                            on='date'
                            )


In [28]:
merge_df.head()

,date,min,max,PRCP,WSPD,WDIR
0,2026-07-27,70,71.1,0,1.964126,255.605381
1,2026-07-28,69.1,77,0,2.410526,216.140351
2,2026-07-29,69.1,77,0,2.615678,209.364407
3,2026-07-30,68,77,0,2.352703,203.963964


In [30]:
full_df = merge_df.merge(water_df,
               how='left',
               on='date'
               )

In [5]:
def fetch_all_data():
    weather_df = fetch_daily_weather()
    water_df = fetch_water_info()
    wind_df = fetch_daily_buoy_data()

    merge_df = weather_df.merge(wind_df,
                            how='left',
                            on='date'
                            )

    full_df = merge_df.merge(water_df,
               how='left',
               on='date'
               )

    return full_df

In [32]:
full_df = full_df.dropna()

In [7]:
DATA_PATH = '../app_data/app-data.parquet'

In [36]:
full_df.to_parquet(DATA_PATH,
                   index=False,
                   engine='pyarrow'
                   )

---

### check for new records and update data file

In [8]:
def fetch_existing_data(data_path = DATA_PATH):
    current_df = pd.read_parquet(data_path)
    current_df['date'] = pd.to_datetime(current_df.date)
    return current_df

In [24]:
current_df = fetch_existing_data()

In [10]:
most_recent_date = current_df.date.max()

In [26]:
current_df.head()

,date,min,max,PRCP,WSPD,WDIR,surf_temp_c,bottom_temp
0,2026-07-30,68,77,0,2.352703,203.963964,21.0,18.2
1,2026-07-31,66.9,69.1,0,1.741176,209.607843,NaN,NaN


In [11]:
current_df.date.max()

Timestamp('2026-07-30 00:00:00')

In [12]:
incoming_records = fetch_all_data()

In [14]:
incoming_records.head()

,date,min,max,PRCP,WSPD,WDIR,surf_temp_c,bottom_temp
0,2026-07-28,70,77,0,2.410526,216.140351,NaN,NaN
1,2026-07-29,69.1,77,0,2.615678,209.364407,NaN,NaN
2,2026-07-30,68,77,0,2.352703,203.963964,21.0,18.2
3,2026-07-31,66.9,69.1,0,1.758586,212.929293,NaN,NaN


In [15]:
## check for updates to the existing data
incoming_dates = set(incoming_records.date.unique())
current_clean_df = current_df[current_df.date.isin(incoming_dates)]

In [16]:
for idx in current_clean_df.index:
    row = current_clean_df.loc[idx]
    current_record = row.to_dict()
    print(current_record)
    date = current_record['date']
    incoming_row = incoming_records[incoming_records.date == date]
    incoming_record_dic = incoming_row.to_dict(orient='records')[0]
    print(incoming_record_dic)
    for key in current_record:
        incoming_value = incoming_record_dic[key]
        if current_record[key] != incoming_value:
            # update the record
            print(f'updating record {date} {key} -> {incoming_value}')

            current_df.loc[idx, key] = incoming_value


{'date': Timestamp('2026-07-30 00:00:00'), 'min': '68', 'max': '77', 'PRCP': 0, 'WSPD': 2.3527027027027025, 'WDIR': 203.96396396396398, 'surf_temp_c': 21.0, 'bottom_temp': 18.2}
{'date': Timestamp('2026-07-30 00:00:00'), 'min': '68', 'max': '77', 'PRCP': 0, 'WSPD': 2.3527027027027025, 'WDIR': 203.96396396396398, 'surf_temp_c': 21.0, 'bottom_temp': 18.2}


In [17]:
new_records_df = incoming_records[incoming_records.date > most_recent_date]

In [18]:
if not new_records_df.empty:
    current_df = pd.concat([current_df, new_records_df])

In [ ]:
current_df.drop_duplicates(subset='date',
                           inplace=True,
                           keep='last'
                           )

In [20]:
current_df.head(2)

,date,min,max,PRCP,WSPD,WDIR,surf_temp_c,bottom_temp
0,2026-07-30,68,77,0,2.352703,203.963964,21.0,18.2
3,2026-07-31,66.9,69.1,0,1.758586,212.929293,NaN,NaN


In [85]:
current_df.to_parquet(DATA_PATH,
                      index=False,
                      engine='pyarrow'
                      )

In [2]:
from app.fetch_data.data_update import update_data

ModuleNotFoundError: No module named 'app'

In [1]:
update_data()

NameError: name 'update_data' is not defined